In [1]:
import pandas as pd
import numpy as np
import sqlite3
import joblib
import os
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import LabelEncoder
os.makedirs("../dashboard", exist_ok=True)
print("All imports successful!")

All imports successful!


In [7]:
# Load data
conn = sqlite3.connect("../data/attrition.db")
df = pd.read_sql("SELECT * FROM employees", conn)
conn.close()

# Load models and weights
lgbm    = joblib.load("../output/improved/lgbm_final.pkl")
gb      = joblib.load("../output/improved/gb_final.pkl")
rf      = joblib.load("../output/improved/rf_final.pkl")
weights = joblib.load("../output/improved/blend_weights.pkl")

w_lgbm = weights["lgbm"]
w_gb   = weights["gb"]
w_rf   = weights["rf"]

# Fix: load final_features correctly — check what's actually in the file
raw = pd.read_csv("../output/final_features.csv")
print("Raw final_features.csv head:")
print(raw.head())
print("Shape:", raw.shape)
print("Columns:", raw.columns.tolist())

# Load correctly based on structure
if raw.shape[1] == 1:
    final_features = raw.iloc[:, 0].tolist()
else:
    final_features = raw.squeeze().tolist()

# Remove any numeric/index artifacts
final_features = [f for f in final_features
                  if isinstance(f, str) and f not in ["0","1","2","3"]]

print(f"\nFinal features ({len(final_features)}): {final_features}")

Raw final_features.csv head:
                   0
0                Age
1      MonthlyIncome
2           OverTime
3  TotalWorkingYears
4  SalaryToRoleRatio
Shape: (7, 1)
Columns: ['0']

Final features (7): ['Age', 'MonthlyIncome', 'OverTime', 'TotalWorkingYears', 'SalaryToRoleRatio', 'SatisfactionComposite', 'DailyRate']


In [8]:
# Encode categoricals
label_cols = ["BusinessTravel", "Department", "EducationField",
              "Gender", "JobRole", "MaritalStatus", "OverTime"]
le = LabelEncoder()
df_enc = df.copy()
for col in label_cols:
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))

# Drop target columns
drop_cols = [c for c in ["Attrition", "Attrition_Binary", "EmployeeNumber",
                          "OverTime_Binary"] if c in df_enc.columns]
X = df_enc.drop(columns=drop_cols)

# Add derived features
role_avg = X.groupby("JobRole")["MonthlyIncome"].transform("mean")
X["SalaryToRoleRatio"]     = (X["MonthlyIncome"] / role_avg).round(3)
X["TenureStabilityScore"]  = (X["YearsAtCompany"] /
                               (X["TotalWorkingYears"] + 1)).round(3)
X["SatisfactionComposite"] = (X["JobSatisfaction"] +
                               X["EnvironmentSatisfaction"] +
                               X["WorkLifeBalance"]) / 3

# DailyRate is already in X from the original dataset — just confirm
print("Features available in X:")
print([f for f in final_features if f in X.columns])
print("\nFeatures missing from X:")
print([f for f in final_features if f not in X.columns])

# Build final feature matrix — all 7 features should now be present
X_final = X[final_features].fillna(X[final_features].median())
print(f"\nX_final shape: {X_final.shape}")
print(f"Nulls: {X_final.isnull().sum().sum()}")
print(f"Columns: {X_final.columns.tolist()}")

# Score all employees
p_lgbm  = lgbm.predict_proba(X_final.values)[:, 1]
p_gb    = gb.predict_proba(X_final.values)[:, 1]
p_rf    = rf.predict_proba(X_final.values)[:, 1]
p_blend = w_lgbm*p_lgbm + w_gb*p_gb + w_rf*p_rf

print(f"\nScored {len(df)} employees successfully")
print(f"Avg risk score:    {p_blend.mean():.3f}")
print(f"High risk (>0.6):  {(p_blend > 0.6).sum()} employees")
print(f"Medium risk:       {((p_blend > 0.3) & (p_blend <= 0.6)).sum()} employees")
print(f"Low risk:          {(p_blend <= 0.3).sum()} employees")

Features available in X:
['Age', 'MonthlyIncome', 'OverTime', 'TotalWorkingYears', 'SalaryToRoleRatio', 'SatisfactionComposite', 'DailyRate']

Features missing from X:
[]

X_final shape: (1470, 7)
Nulls: 0
Columns: ['Age', 'MonthlyIncome', 'OverTime', 'TotalWorkingYears', 'SalaryToRoleRatio', 'SatisfactionComposite', 'DailyRate']

Scored 1470 employees successfully
Avg risk score:    0.320
High risk (>0.6):  123 employees
Medium risk:       525 employees
Low risk:          822 employees


In [10]:
# Start with original readable columns (not encoded — keep text for Tableau)
tableau_df = df.copy()

# Add model outputs
tableau_df["attrition_probability"] = p_blend.round(4)
tableau_df["risk_tier"] = pd.cut(
    p_blend,
    bins=[0, 0.3, 0.6, 1.0],
    labels=["Low", "Medium", "High"]
).astype(str)

# Cost columns
tableau_df["replacement_cost_usd"] = (
    tableau_df["MonthlyIncome"] * 12 * 0.5
).round(0)

tableau_df["at_risk_cost"] = np.where(
    tableau_df["risk_tier"] == "High",
    tableau_df["replacement_cost_usd"], 0
)

# Binary attrition
tableau_df["Attrition_Binary"] = (tableau_df["Attrition"] == "Yes").astype(int)

# Grouping bands
tableau_df["IncomeBand"] = pd.cut(
    tableau_df["MonthlyIncome"],
    bins=[0, 3000, 6000, 10000, 20000],
    labels=["<3K", "3K-6K", "6K-10K", "10K+"]
).astype(str)

tableau_df["TenureBand"] = pd.cut(
    tableau_df["YearsAtCompany"],
    bins=[-1, 2, 5, 10, 40],
    labels=["0-2 yrs", "3-5 yrs", "6-10 yrs", "10+ yrs"]
).astype(str)

tableau_df["SatisfactionTier"] = pd.cut(
    tableau_df["JobSatisfaction"],
    bins=[0, 2, 3, 4],
    labels=["Low (1-2)", "Medium (3)", "High (4)"]
).astype(str)

print(f"tableau_df shape: {tableau_df.shape}")
print(f"\nRisk tier distribution:")
print(tableau_df["risk_tier"].value_counts())
print(f"\nSample columns added:")
print(tableau_df[["attrition_probability","risk_tier",
                   "replacement_cost_usd","IncomeBand","TenureBand"]].head(3))

tableau_df shape: (1470, 41)

Risk tier distribution:
risk_tier
Low       822
Medium    525
High      123
Name: count, dtype: int64

Sample columns added:
   attrition_probability risk_tier  replacement_cost_usd IncomeBand TenureBand
0                 0.4399    Medium               35958.0      3K-6K   6-10 yrs
1                 0.3077    Medium               30780.0      3K-6K   6-10 yrs
2                 0.4709    Medium               12540.0        <3K    0-2 yrs


In [11]:
# --- File 1: Master employee table (main data source) ---
tableau_df.to_csv("../dashboard/employees_master.csv", index=False)
print("Saved employees_master.csv")

# --- File 2: Department summary ---
dept_summary = tableau_df.groupby("Department").agg(
    total_employees     = ("EmployeeNumber", "count"),
    attrition_count     = ("Attrition_Binary", "sum"),
    attrition_rate_pct  = ("Attrition_Binary", lambda x: round(x.mean()*100, 1)),
    avg_monthly_income  = ("MonthlyIncome", lambda x: round(x.mean(), 0)),
    replacement_cost    = ("replacement_cost_usd", "sum"),
    at_risk_cost        = ("at_risk_cost", "sum"),
    high_risk_count     = ("risk_tier", lambda x: (x=="High").sum()),
    avg_risk_score      = ("attrition_probability", lambda x: round(x.mean(), 3))
).reset_index()
dept_summary.to_csv("../dashboard/dept_summary.csv", index=False)
print("Saved dept_summary.csv")

# --- File 3: Job role summary ---
role_summary = tableau_df.groupby(["Department", "JobRole"]).agg(
    total_employees    = ("EmployeeNumber", "count"),
    attrition_rate_pct = ("Attrition_Binary", lambda x: round(x.mean()*100, 1)),
    avg_risk_score     = ("attrition_probability", lambda x: round(x.mean(), 3)),
    high_risk_count    = ("risk_tier", lambda x: (x=="High").sum()),
    avg_income         = ("MonthlyIncome", lambda x: round(x.mean(), 0))
).reset_index()
role_summary.to_csv("../dashboard/role_summary.csv", index=False)
print("Saved role_summary.csv")

# --- File 4: Attrition drivers (for bar charts) ---
drivers = pd.DataFrame([
    # Overtime effect
    {"Driver": "OverTime",    "Segment": "Yes",
     "Attrition_Rate": round(tableau_df[tableau_df["OverTime"]=="Yes"]["Attrition_Binary"].mean()*100,1)},
    {"Driver": "OverTime",    "Segment": "No",
     "Attrition_Rate": round(tableau_df[tableau_df["OverTime"]=="No"]["Attrition_Binary"].mean()*100,1)},
    # Job satisfaction
    {"Driver": "JobSatisfaction", "Segment": "Low (1-2)",
     "Attrition_Rate": round(tableau_df[tableau_df["JobSatisfaction"]<=2]["Attrition_Binary"].mean()*100,1)},
    {"Driver": "JobSatisfaction", "Segment": "High (3-4)",
     "Attrition_Rate": round(tableau_df[tableau_df["JobSatisfaction"]>=3]["Attrition_Binary"].mean()*100,1)},
    # Tenure bands
    *[{"Driver": "TenureBand", "Segment": band,
       "Attrition_Rate": round(tableau_df[tableau_df["TenureBand"]==band]["Attrition_Binary"].mean()*100,1)}
      for band in ["0-2 yrs", "3-5 yrs", "6-10 yrs", "10+ yrs"]],
    # Income bands
    *[{"Driver": "IncomeBand", "Segment": band,
       "Attrition_Rate": round(tableau_df[tableau_df["IncomeBand"]==band]["Attrition_Binary"].mean()*100,1)}
      for band in ["<3K", "3K-6K", "6K-10K", "10K+"]],
])
drivers.to_csv("../dashboard/attrition_drivers.csv", index=False)
print("Saved attrition_drivers.csv")

# --- File 5: Risk distribution for scatter plot ---
scatter_df = tableau_df[[
    "EmployeeNumber", "Department", "JobRole", "MonthlyIncome",
    "YearsAtCompany", "OverTime", "JobSatisfaction",
    "attrition_probability", "risk_tier", "Attrition",
    "replacement_cost_usd", "TenureBand", "IncomeBand"
]].copy()
scatter_df.to_csv("../dashboard/risk_scatter.csv", index=False)
print("Saved risk_scatter.csv")

# --- File 6: Cross-dataset comparison ---
cross = pd.DataFrame([
    {"Dataset": "IBM (Training)",     "Rows": 1470,  "Attrition_Rate": 16.1, "AUC": 0.77, "Industry": "Technology"},
    {"Dataset": "HR Analytics",       "Rows": 14999, "Attrition_Rate": 23.8, "AUC": 0.63, "Industry": "Service"},
    {"Dataset": "MFG (Resignation)",  "Rows": 6284,  "Attrition_Rate": 6.1,  "AUC": 0.66, "Industry": "Manufacturing"},
])
cross.to_csv("../dashboard/cross_dataset_summary.csv", index=False)
print("Saved cross_dataset_summary.csv")

print("\nAll 6 dashboard files saved to dashboard/ folder")
print("\nFiles:")
for f in sorted(os.listdir("../dashboard")):
    size = os.path.getsize(f"../dashboard/{f}") / 1024
    print(f"  {f}  ({size:.0f} KB)")

Saved employees_master.csv
Saved dept_summary.csv
Saved role_summary.csv
Saved attrition_drivers.csv
Saved risk_scatter.csv
Saved cross_dataset_summary.csv

All 6 dashboard files saved to dashboard/ folder

Files:
  attrition_drivers.csv  (0 KB)
  cross_dataset_summary.csv  (0 KB)
  dept_summary.csv  (0 KB)
  employees_master.csv  (288 KB)
  risk_scatter.csv  (132 KB)
  role_summary.csv  (1 KB)
